In [9]:
import MDAnalysis as mda
import prolif as plf
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning) # This is for ProLIF because it generates a ton of warnings on AMBER trajs

In [10]:
traj = "test/a1_rc3_RBD_traj.pdb"

In [11]:
u = mda.Universe(traj)
chain_data = []
for segment in u.segments:
    # Extract the ordered residue names for the current segment
    residues = segment.residues
    sequence = "-".join([res.resname for res in residues])
    
    # Store the metadata
    chain_data.append({
        "Chain_ID": segment.segid,
        "Residue_Count": len(residues),
        "Sequence": sequence
    })
    
df_chains = pd.DataFrame(chain_data)
print(f"Total Atoms: {len(u.atoms)} | Total Residues: {len(u.residues)}")
display(df_chains)

Total Atoms: 3622 | Total Residues: 232


,Chain_ID,Residue_Count,Sequence
0,SYSTEM,232,ACE-CYS-PRO-PHE-GLY-GLU-VAL-PHE-ASN-ALA-THR-AR...


In [12]:
# Selection
receptor = "protein"
interactions = [
    "VdWContact", "HBDonor", "HBAcceptor", 
    "PiStacking", "EdgeToFace", "FaceToFace", 
    "Anionic", "Cationic", "CationPi"
]

In [7]:
u = mda.Universe(traj)
protein_ag = u.select_atoms(receptor)
fp = plf.Fingerprint(interactions, vicinity_cutoff=5.0)

# Avoid cero-sum when PROLIF calculates self-self interactions
for name, interaction_func in fp.interactions.items():
    def make_safe_interaction(orig_func):
        def safe_interaction(lig_res, prot_res, **kwargs):
            # Prevent a residue from being evaluated against itself
            if lig_res.resid == prot_res.resid: 
                return None
            return orig_func(lig_res, prot_res, **kwargs)
        return safe_interaction
    fp.interactions[name] = make_safe_interaction(interaction_func)

# Intramolecular interactions
fp.run(u.trajectory, lig=protein_ag, prot=protein_ag)
df_raw = fp.to_dataframe()

# Frequency calculation
df_freq = (df_raw.mean() * 100).to_frame(name="Frequency_%")
# Column names
df_freq.index.names = ['Receptor_Residue_1', 'Receptor_Residue_2', 'Interaction']
# Snippet
display(df_freq.head(15))

Initializing Universe and configuring ProLIF...
Running intramolecular ProLIF analysis in memory...


  0%|          | 0/50 [00:00<?, ?it/s]

Calculating interaction frequencies...
Analysis complete.


Frequency_%
Receptor_Residue_1 Receptor_Residue_2 Interaction             
ACE1               CYS2               VdWContact         100.0
                   CYS27              VdWContact           4.0
                   VAL28              VdWContact          24.0
CYS2               ACE1               VdWContact         100.0
                   PRO3               VdWContact         100.0
                   PHE4               VdWContact         100.0
                                      HBAcceptor          88.0
                   ILE24              VdWContact          40.0
                   CYS27              VdWContact          62.0
                   VAL28              VdWContact          92.0
                                      HBDonor             86.0
                   ALA29              VdWContact          30.0
PRO3               CYS2               VdWContact         100.0
                   PHE4               VdWContact         100.0
                   GLY5               VdWContact          12.0

In [8]:
# Output
output_csv_path = "test/intra_test.csv"
df_freq.to_csv(output_csv_path)